<a href="https://colab.research.google.com/github/fmaignacio/observatorio-tere/blob/main/organiza_transcricoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import pandas as pd
from google.colab import drive
from datetime import datetime

# 1. Monte seu Google Drive
drive.mount('/content/drive')

# --- CONFIGURAÇÃO ---
pasta_transcricoes = '/content/drive/MyDrive/observatorio_teresopolis/TranscriçõesSessoesTeresopolis'

LISTA_MESTRA_VEREADORES = [
    "Amanda", "André do Gás", "Bruninho Almeida", "Cacau Repórter", "Caio Perfister",
    "Calé", "Dudu do Resgate", "Diego Barbosa", "Fabinho Filé", "Fidel Faria",
    "Igor Faraco", "João Miguel", "Luciano Santos", "Márcia Valentim", "Marcos Rangel",
    "Maurício Lopes", "Paulinho Nogueira", "Amurim", "Sandrinho", "Totó", "Vitinho Nogueira",
    "Érica Marra", "Totó Online"
]

dados_extraidos = []
print(f"🔍 Iniciando análise COMPLETA com extração de EMENTAS e LINKS\n")

# --- FUNÇÕES AUXILIARES ---

def extrair_url_youtube(texto):
    """Extrai a URL do YouTube do arquivo de transcrição"""
    match = re.search(r'https://www\.youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})', texto)
    if match:
        return match.group(0)
    return None

def extrair_ementa(texto_completo, pl_numero, posicao_mencao):
    """
    Extrai a ementa/descrição do PL
    """
    # Pega contexto ao redor da menção do PL
    inicio = max(0, posicao_mencao - 100)
    fim = min(len(texto_completo), posicao_mencao + 1000)
    contexto = texto_completo[inicio:fim]

    # Padrões de ementa mais flexíveis e abrangentes
    padroes_ementa = [
        # Padrão 1: "dispõe sobre..." (o mais comum)
        r'dispõe\s+sobre\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 2: "institui..."
        r'institui\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 3: "autoriza..."
        r'autoriza\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 4: "cria..."
        r'cria\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 5: "revoga..."
        r'revoga\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 6: "denomina" ou "denominação"
        r'(?:dispõe\s+sobre\s+)?denomina(?:ção)?\s+de\s+logradouro\s+público',

        # Padrão 7: "altera..."
        r'altera\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 8: "inclui..."
        r'inclui\s+([^.]+?)(?:\s+e\s+d[aáà]\s+outras\s+providências)?\.?',

        # Padrão 9: Pega qualquer coisa após "de autoria" até o ponto
        r'de\s+autoria[^.]*?\.\s*([A-Z][^.]+?\.)(?:\s+e\s+d[aáà]\s+outras\s+providências)?',
    ]

    for padrao in padroes_ementa:
        match = re.search(padrao, contexto, re.IGNORECASE | re.DOTALL)
        if match:
            # Captura o texto completo do match
            ementa_bruta = match.group(0)

            # Limpa a ementa
            ementa = ementa_bruta.strip()
            ementa = re.sub(r'\s+', ' ', ementa)  # Remove espaços múltiplos
            ementa = re.sub(r'\n+', ' ', ementa)  # Remove quebras de linha

            # Limita tamanho
            if len(ementa) > 300:
                ementa = ementa[:300] + "..."

            # Capitaliza primeira letra
            if ementa and len(ementa) > 0:
                ementa = ementa[0].upper() + ementa[1:]

            return ementa

    return "Ementa não identificada"

def verificar_presenca(texto_chamada, lista_mestral):
    presentes = [vereador for vereador in lista_mestral
                 if re.search(r'\b' + re.escape(vereador) + r'\b', texto_chamada, re.IGNORECASE)]
    return presentes

def extrair_data_sessao(texto):
    match = re.search(
        r'(\d{1,2})\s+de\s+(janeiro|fevereiro|março|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)\s+de\s+(\d{4})',
        texto,
        re.IGNORECASE
    )
    if match:
        dia, mes_nome, ano = match.groups()
        meses = {
            'janeiro': 1, 'fevereiro': 2, 'março': 3, 'abril': 4, 'maio': 5, 'junho': 6,
            'julho': 7, 'agosto': 8, 'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12
        }
        mes_num = meses[mes_nome.lower()]
        return f"{int(ano)}-{mes_num:02d}-{int(dia):02d}"
    return None

def extrair_autor_robusto(contexto_pl):
    """
    Extração mais robusta do autor
    """
    # Padrão 1: "de autoria do vereador X"
    match1 = re.search(r'de\s+autoria\s+do\s+vereador(?:a)?\s+([\w\s]+?)(?:\s*,|\s*dispõe|\s*institui|\s*autoriza|\s*cria|\s*\n)',
                       contexto_pl, re.IGNORECASE)
    if match1:
        autor = match1.group(1).strip()
        # Limpar nomes compostos
        autor = re.sub(r'\s+', ' ', autor)
        return autor

    # Padrão 2: Procurar nomes da lista mestra no contexto
    for nome_oficial, variacoes in {
        "Amanda": ["Amanda", "professora Amanda"],
        "André do Gás": ["André do Gás", "André"],
        "Bruninho Almeida": ["Bruninho Almeida", "Bruninho"],
        "Cacau Repórter": ["Cacau Repórter", "Cacau"],
        "Caio Perfister": ["Caio Perfister", "Caio Perfiste", "Caio"],
        "Calé": ["Calé"],
        "Dudu do Resgate": ["Dudu do Resgate", "Dudo Resgate", "Dudu"],
        "Diego Barbosa": ["Diego Barbosa", "Diego"],
        "Fabinho Filé": ["Fabinho Filé", "Fabinho"],
        "Fidel Faria": ["Fidel Faria", "Fidel"],
        "Igor Faraco": ["Igor Faraco", "Igor"],
        "João Miguel": ["João Miguel", "João"],
        "Luciano Santos": ["Luciano Santos", "Luciano"],
        "Márcia Valentim": ["Márcia Valentim", "Márcia", "Marcia Valentin"],
        "Marcos Rangel": ["Marcos Rangel", "Rangel"],
        "Maurício Lopes": ["Maurício Lopes", "Maurício"],
        "Paulinho Nogueira": ["Paulinho Nogueira", "Paulinho"],
        "Sandrinho": ["Sandrinho"],
        "Totó": ["Totó", "Totó Online"],
        "Vitinho Nogueira": ["Vitinho Nogueira", "Vitinho", "Vitim Nogueira", "Vitin Nogueira"],
        "Érica Marra": ["Érica Marra", "Érica"]
    }.items():
        for variacao in variacoes:
            if re.search(r'\b' + re.escape(variacao) + r'\b', contexto_pl, re.IGNORECASE):
                return nome_oficial

    return "Autor não identificado"

# --- SCRIPT PRINCIPAL ---

try:
    arquivos_processados = 0
    pls_total = 0
    pls_com_ementa = 0

    arquivos = [f for f in os.listdir(pasta_transcricoes) if f.endswith('.txt')]
    print(f"📂 Processando {len(arquivos)} arquivos...\n")

    for nome_arquivo in arquivos:
        caminho_completo = os.path.join(pasta_transcricoes, nome_arquivo)
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            conteudo = f.read()

        data_sessao = extrair_data_sessao(conteudo)
        url_youtube = extrair_url_youtube(conteudo)

        bloco_chamada_match = re.search(
            r"chamada dos vereadores\.(.*?)(Questão de ordem|Peço para que todos)",
            conteudo,
            re.DOTALL | re.IGNORECASE
        )
        vereadores_presentes = verificar_presenca(
            bloco_chamada_match.group(1) if bloco_chamada_match else "",
            LISTA_MESTRA_VEREADORES
        )

        # PADRÃO MAIS FLEXÍVEL PARA CAPTURAR PLs
        # Captura: "Projeto de Lei" + número/ano
        mencoes_pl = re.finditer(
            r'Projeto\s+de\s+[Ll]ei\s+(?:n[úº]mero|n[úº]|nº)?\s*(\d{1,3}\/\d{4})',
            conteudo,
            re.IGNORECASE
        )

        pls_processados_arquivo = set()  # Evita duplicatas no mesmo arquivo

        for mencao in mencoes_pl:
            pl = mencao.group(1)

            # Evita processar o mesmo PL múltiplas vezes no mesmo arquivo
            if pl in pls_processados_arquivo:
                continue

            pls_processados_arquivo.add(pl)
            pls_total += 1

            posicao_mencao = mencao.start()

            # Contexto para extração de autor e ementa
            inicio_contexto = max(0, posicao_mencao - 50)
            fim_contexto = min(len(conteudo), posicao_mencao + 1000)
            contexto_autor = conteudo[inicio_contexto:fim_contexto]

            # Extrai autor
            autor = extrair_autor_robusto(contexto_autor)

            # Extrai ementa
            ementa = extrair_ementa(conteudo, pl, posicao_mencao)
            if ementa != "Ementa não identificada":
                pls_com_ementa += 1

            # Determina status
            contexto_status = conteudo[posicao_mencao : posicao_mencao + 2000].lower()

            status_votacao = 'Não identificado'
            if re.search(r"em votação.*?aprovado", contexto_status):
                status_votacao = 'Aprovado (Votação Simbólica)'
            elif re.search(r"encaminhado.*?comissões|parecer favorável", contexto_status):
                status_votacao = 'Encaminhado para Comissão'
            elif "em discussão" in contexto_status:
                status_votacao = 'Em Discussão'
            elif 'rejeitado' in contexto_status:
                status_votacao = 'Rejeitado'

            dados_extraidos.append({
                'Data Sessão': data_sessao,
                'PL': pl,
                'Autor': autor,
                'Ementa': ementa,
                'Status': status_votacao,
                'Votos': "N/A",
                'Presentes': ", ".join(vereadores_presentes) if vereadores_presentes else "Chamada não identificada",
                'Fonte': nome_arquivo,
                'Link YouTube': url_youtube if url_youtube else "N/A"
            })

        arquivos_processados += 1
        if arquivos_processados % 10 == 0:
            print(f"✅ Processados {arquivos_processados}/{len(arquivos)} arquivos...")

    if dados_extraidos:
        df_final = pd.DataFrame(dados_extraidos).drop_duplicates(subset=['Fonte', 'PL']).reset_index(drop=True)

        df_final['Data Sessão'] = pd.to_datetime(df_final['Data Sessão'], errors='coerce')

        # Remove datas inválidas (muito antigas)
        df_final = df_final[df_final['Data Sessão'] >= '2024-01-01']

        df_final = df_final.sort_values(by=['PL', 'Data Sessão'], ascending=[True, True])

        # Estatísticas finais
        total_pls = len(df_final)
        pls_com_ementa_final = len(df_final[df_final['Ementa'] != 'Ementa não identificada'])
        pls_com_link = len(df_final[df_final['Link YouTube'] != 'N/A'])
        percentual_ementas = (pls_com_ementa_final / total_pls) * 100 if total_pls > 0 else 0
        percentual_links = (pls_com_link / total_pls) * 100 if total_pls > 0 else 0

        print("\n" + "="*80)
        print("✅ ANÁLISE COMPLETA CONCLUÍDA!")
        print("="*80)
        print(f"📊 Estatísticas:")
        print(f"   📂 Arquivos processados: {arquivos_processados}")
        print(f"   📋 Total de registros: {total_pls}")
        print(f"   📝 PLs com ementa: {pls_com_ementa_final} ({percentual_ementas:.1f}%)")
        print(f"   🎥 PLs com link YouTube: {pls_com_link} ({percentual_links:.1f}%)")
        print(f"   📅 Período: {df_final['Data Sessão'].min():%d/%m/%Y} até {df_final['Data Sessão'].max():%d/%m/%Y}")
        print(f"   🏛️ PLs únicos: {df_final['PL'].nunique()}")

        print("\n🔍 Verificando PL 197/2025:")
        pl_197 = df_final[df_final['PL'] == '197/2025']
        if not pl_197.empty:
            print("   ✅ PL 197/2025 ENCONTRADO!")
            for idx, row in pl_197.iterrows():
                print(f"      📅 Data: {row['Data Sessão']:%d/%m/%Y}")
                print(f"      👤 Autor: {row['Autor']}")
                print(f"      📝 Ementa: {row['Ementa'][:100]}...")
                print(f"      📊 Status: {row['Status']}")
        else:
            print("   ❌ PL 197/2025 NÃO encontrado")

        print("\n📋 Amostra de PLs extraídos:")
        for idx, row in df_final.head(10).iterrows():
            print(f"\n   PL {row['PL']} - {row['Autor']}")
            if row['Ementa'] != 'Ementa não identificada':
                print(f"   📝 {row['Ementa'][:80]}...")
            if row['Link YouTube'] != 'N/A':
                print(f"   🎥 {row['Link YouTube']}")

        display(df_final)

        # Salva os arquivos
        caminho_csv = '/content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.csv'
        caminho_excel = '/content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.xlsx'

        df_final.to_csv(caminho_csv, index=False)

        with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
            df_final.to_excel(writer, sheet_name='PLs', index=False)

        print(f"\n💾 Arquivos salvos:")
        print(f"   📄 {caminho_csv}")
        print(f"   📊 {caminho_excel}")

    else:
        print("\n❌ Nenhum dado foi extraído")

except FileNotFoundError:
    print(f"❌ ERRO: A pasta '{pasta_transcricoes}' não foi encontrada.")
except Exception as e:
    print(f"❌ Erro inesperado: {e}")
    import traceback
    traceback.print_exc()

print("\n✅ PROCESSAMENTO CONCLUÍDO!")

Mounted at /content/drive
🔍 Iniciando análise COMPLETA com extração de EMENTAS e LINKS

📂 Processando 80 arquivos...

✅ Processados 10/80 arquivos...
✅ Processados 20/80 arquivos...
✅ Processados 30/80 arquivos...
✅ Processados 40/80 arquivos...
✅ Processados 50/80 arquivos...
✅ Processados 60/80 arquivos...
✅ Processados 70/80 arquivos...
✅ Processados 80/80 arquivos...

✅ ANÁLISE COMPLETA CONCLUÍDA!
📊 Estatísticas:
   📂 Arquivos processados: 80
   📋 Total de registros: 398
   📝 PLs com ementa: 395 (99.2%)
   🎥 PLs com link YouTube: 315 (79.1%)
   📅 Período: 23/02/2024 até 09/12/2025
   🏛️ PLs únicos: 246

🔍 Verificando PL 197/2025:
   ❌ PL 197/2025 NÃO encontrado

📋 Amostra de PLs extraídos:

   PL 003/2025 - Cacau Repórter
   📝 Dispõe sobre a...
   🎥 https://www.youtube.com/watch?v=KOFfnMQI7aA

   PL 004/2025 - Amanda
   📝 Inclui o...
   🎥 https://www.youtube.com/watch?v=1_fKnlVY00Q

   PL 005/2025 - Vitim Nogueira
   📝 Dispõe sobre d...
   🎥 https://www.youtube.com/watch?v=KOFfnMQI

,Data Sessão,PL,Autor,Ementa,Status,Votos,Presentes,Fonte,Link YouTube
314,2025-03-08,003/2025,Cacau Repórter,Dispõe sobre a,Não identificado,N/A,"Amanda, André do Gás, Cacau Repórter, Caio Per...",video-KOFfnMQI7aA-ytranscript.txt,https://www.youtube.com/watch?v=KOFfnMQI7aA
457,2025-12-02,004/2025,Amanda,Inclui o,Não identificado,N/A,Chamada não identificada,Sessão da Câmara de Teresópolis - 02 - 12 -...,https://www.youtube.com/watch?v=1_fKnlVY00Q
315,2025-03-08,005/2025,Vitim Nogueira,Dispõe sobre d,Não identificado,N/A,"Amanda, André do Gás, Cacau Repórter, Caio Per...",video-KOFfnMQI7aA-ytranscript.txt,https://www.youtube.com/watch?v=KOFfnMQI7aA
316,2025-03-08,006/2025,Vitim Nogueira,Dispõe sobre d,Não identificado,N/A,"Amanda, André do Gás, Cacau Repórter, Caio Per...",video-KOFfnMQI7aA-ytranscript.txt,https://www.youtube.com/watch?v=KOFfnMQI7aA
283,2025-04-02,015/2025,Dudo Resgate,Dispõe sobre a,Não identificado,N/A,"Amanda, Bruninho Almeida, Cacau Repórter, Caio...",video-q4OpqYXvxh0-ytranscript.txt,https://www.youtube.com/watch?v=q4OpqYXvxh0
...,...,...,...,...,...,...,...,...,...
276,2025-04-02,66/2025,Dudu do Resgate,Dispõe sobre r,Aprovado (Votação Simbólica),N/A,"Amanda, Bruninho Almeida, Cacau Repórter, Caio...",video-q4OpqYXvxh0-ytranscript.txt,https://www.youtube.com/watch?v=q4OpqYXvxh0
279,2025-04-02,71/2025,Dudu do Resgate,Dispõe sobre a,Em Discussão,N/A,"Amanda, Bruninho Almeida, Cacau Repórter, Caio...",video-q4OpqYXvxh0-ytranscript.txt,https://www.youtube.com/watch?v=q4OpqYXvxh0
280,2025-04-02,72/2025,Autor não identificado,Autoriza o,Aprovado (Votação Simbólica),N/A,"Amanda, Bruninho Almeida, Cacau Repórter, Caio...",video-q4OpqYXvxh0-ytranscript.txt,https://www.youtube.com/watch?v=q4OpqYXvxh0
216,2025-04-15,75/2025,Cacau Repórter,Dispõe sobre o,Não identificado,N/A,"Amanda, Bruninho Almeida, Cacau Repórter, Caio...",video-GWiUIy5IE0Q-ytranscript.txt,https://www.youtube.com/watch?v=GWiUIy5IE0Q



💾 Arquivos salvos:
   📄 /content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.csv
   📊 /content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.xlsx

✅ PROCESSAMENTO CONCLUÍDO!
